<a href="https://colab.research.google.com/github/Zyu-Peng/learning_code/blob/AF2/batch/AlphaFold2_batch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#ColabFold v1.6.0: AlphaFold2 w/ MMseqs2 BATCH

<img src="https://raw.githubusercontent.com/sokrypton/ColabFold/main/.github/ColabFold_Marv_Logo_Small.png" height="256" align="right" style="height:256px">

Easy to use AlphaFold2 protein structure [(Jumper et al. 2021)](https://www.nature.com/articles/s41586-021-03819-2) and complex [(Evans et al. 2021)](https://www.biorxiv.org/content/10.1101/2021.10.04.463034v1) prediction using multiple sequence alignments generated through MMseqs2. For details, refer to our manuscript:

[Mirdita M, Schütze K, Moriwaki Y, Heo L, Ovchinnikov S, Steinegger M. ColabFold: Making protein folding accessible to all.
*Nature Methods*, 2022](https://www.nature.com/articles/s41592-022-01488-1)

**Usage**

`input_dir` directory with only fasta files or MSAs stored in Google Drive. MSAs need to be A3M formatted and have an `.a3m` extention. For MSAs MMseqs2 will not be called.

`result_dir` results will be written to the result directory in Google Drive

Old versions: [v1.4](https://colab.research.google.com/github/sokrypton/ColabFold/blob/v1.4.0/batch/AlphaFold2_batch.ipynb), [v1.5.1](https://colab.research.google.com/github/sokrypton/ColabFold/blob/v1.5.1/batch/AlphaFold2_batch.ipynb), [v1.5.2](https://colab.research.google.com/github/sokrypton/ColabFold/blob/v1.5.2/batch/AlphaFold2_batch.ipynb), [v1.5.3-patch](https://colab.research.google.com/github/sokrypton/ColabFold/blob/56c72044c7d51a311ca99b953a71e552fdc042e1/batch/AlphaFold2_batch.ipynb)

<strong>For more details, see <a href="#Instructions">bottom</a> of the notebook and checkout the [ColabFold GitHub](https://github.com/sokrypton/ColabFold). </strong>

-----------

### News
- <b><font color='green'>2023/07/31: The ColabFold MSA server is back to normal. It was using older DB (UniRef30 2202/PDB70 220313) from 27th ~8:30 AM CEST to 31st ~11:10 AM CEST.</font></b>
- <b><font color='green'>2023/06/12: New databases! UniRef30 updated to 2023_02 and PDB to 230517. We now use PDB100 instead of PDB70 (see notes in the [main](https://colabfold.com) notebook).</font></b>
- <b><font color='green'>2023/06/12: We introduced a new default pairing strategy: Previously, for multimer predictions with more than 2 chains, we only pair if all sequences taxonomically match ("complete" pairing). The new default "greedy" strategy pairs any taxonomically matching subsets.</font></b>

In [1]:
#@title Mount google drive
from google.colab import drive
drive.mount('/content/drive')
from sys import version_info
python_version = f"{version_info.major}.{version_info.minor}"

Mounted at /content/drive


In [4]:
#@title Input protein sequence, then hit `Runtime` -> `Run all`

input_dir = '/content/drive/MyDrive/input_fasta' #@param {type:"string"}
result_dir = '/content/drive/MyDrive/result' #@param {type:"string"}

# number of models to use
#@markdown ---
#@markdown ### Advanced settings
msa_mode = "MMseqs2 (UniRef+Environmental)" #@param ["MMseqs2 (UniRef+Environmental)", "MMseqs2 (UniRef only)","single_sequence","custom"]
num_models = 5 #@param [1,2,3,4,5] {type:"raw"}
num_recycles = 3 #@param [1,3,6,12,24,48] {type:"raw"}
stop_at_score = 100 #@param {type:"string"}
#@markdown - early stop computing models once score > threshold (avg. plddt for "structures" and ptmscore for "complexes")
use_custom_msa = False
num_relax = 0 #@param [0, 1, 5] {type:"raw"}
use_amber = num_relax > 0
relax_max_iterations = 200 #@param [0,200,2000] {type:"raw"}
use_templates = False #@param {type:"boolean"}
do_not_overwrite_results = True #@param {type:"boolean"}
zip_results = False #@param {type:"boolean"}


In [5]:
#@title Install dependencies
%%bash -s $use_amber $use_templates $python_version

set -e

USE_AMBER=$1
USE_TEMPLATES=$2
PYTHON_VERSION=$3

if [ ! -f COLABFOLD_READY ]; then
  # install dependencies
  # We have to use "--no-warn-conflicts" because colab already has a lot preinstalled with requirements different to ours
  pip install -q --no-warn-conflicts "colabfold[alphafold-minus-jax] @ git+https://github.com/sokrypton/ColabFold"
  if [ -n "${TPU_NAME}" ]; then
    pip install -q --no-warn-conflicts -U dm-haiku==0.0.10 jax==0.3.25
  fi
  ln -s /usr/local/lib/python3.*/dist-packages/colabfold colabfold
  ln -s /usr/local/lib/python3.*/dist-packages/alphafold alphafold
  # hack to fix TF crash
  rm -f /usr/local/lib/python3.*/dist-packages/tensorflow/core/kernels/libtfkernel_sobol_op.so
  touch COLABFOLD_READY
fi

# Download params (~1min)
python -m colabfold.download

# setup conda
if [ ${USE_AMBER} == "True" ] || [ ${USE_TEMPLATES} == "True" ]; then
  if [ ! -f CONDA_READY ]; then
    wget -qnc https://github.com/conda-forge/miniforge/releases/download/25.3.1-0/Miniforge3-25.3.1-0-Linux-x86_64.sh
    bash Miniforge3-25.3.1-0-Linux-x86_64.sh -bfp /usr/local 2>&1 1>/dev/null
    rm Miniforge3-25.3.1-0-Linux-x86_64.sh
    conda config --set auto_update_conda false
    touch CONDA_READY
  fi
fi
# setup template search
if [ ${USE_TEMPLATES} == "True" ] && [ ! -f HH_READY ]; then
  conda install -y -q -c conda-forge -c bioconda kalign2=2.04 hhsuite=3.3.0 python="${PYTHON_VERSION}" 2>&1 1>/dev/null
  touch HH_READY
fi
# setup openmm for amber refinement
if [ ${USE_AMBER} == "True" ] && [ ! -f AMBER_READY ]; then
  conda install -y -q -c conda-forge openmm=8.2.0 python="${PYTHON_VERSION}" pdbfixer 2>&1 1>/dev/null
  touch AMBER_READY
fi

In [6]:
import os
import glob
import shutil
import torch
from colabfold.batch import get_queries, run
from colabfold.download import default_data_dir
from colabfold.utils import setup_logging
from pathlib import Path

# ==========================================
# 1. 路径设置 (请根据你的实际路径修改)
# ==========================================
input_dir = "/root/autodl-tmp/pengzhengyu/GotenNetToxi/data/fasta_input"
result_dir = "/root/autodl-tmp/pengzhengyu/GotenNetToxi/data/pdb_output"

# 确保输出目录存在
os.makedirs(result_dir, exist_ok=True)

# ==========================================
# 2. 执行 ColabFold 预测
# ==========================================
setup_logging(Path(result_dir).joinpath("log.txt"))
queries, is_complex = get_queries(input_dir)

print(f"开始预测，共有 {len(queries)} 个任务...")

run(
    queries=queries,
    result_dir=result_dir,
    use_templates=False,       # 离线环境通常设为 False
    num_relax=1,               # 设为 1 以获得更好的结构质量
    relax_max_iterations=200,
    msa_mode="mmseqs2_uniref_env", # 如果是完全离线需改为 "single_sequence"
    model_type="auto",
    num_models=5,
    num_recycles=3,
    model_order=[1, 2, 3, 4, 5],
    is_complex=is_complex,
    data_dir=default_data_dir,
    keep_existing_results=True,
    rank_by="auto",
    zip_results=False,         # 不压缩，方便后续处理
    user_agent="colabfold/google-colab-batch",
)

# ==========================================
# 3. 核心整理：提取 PDB 并删除中间文件
# ==========================================
def finalize_and_cleanup(queries, result_dir):
    print("\n" + "="*40)
    print("[Finalizing] 正在执行“一序列一 PDB”整理并清理空间...")

    output_count = 0
    for query in queries:
        jobname = query[0]  # 序列 ID

        # 1. 寻找评分最高 (rank_001) 的 PDB
        search_pattern = os.path.join(result_dir, f"{jobname}_*_rank_001*.pdb")
        matches = glob.glob(search_pattern)

        if matches:
            # 优先选择 relaxed 结构，其次 unrelaxed
            best_pdb = sorted(matches, key=lambda x: "relaxed" in x, reverse=True)[0]

            target_name = f"{jobname}.pdb"
            target_path = os.path.join(result_dir, target_name)

            # 复制并重命名为 ID.pdb
            shutil.copy2(best_pdb, target_path)
            print(f"✅ 提取成功: {target_name}")
            output_count += 1

            # 2. 清理该 ID 下的所有中间文件 (a3m, json, png, 其他 rank 的 pdb)
            all_intermediate_files = glob.glob(os.path.join(result_dir, f"{jobname}_*"))
            for f in all_intermediate_files:
                try:
                    # 绝对不要删掉我们刚刚生成的 target_path
                    if os.path.abspath(f) != os.path.abspath(target_path):
                        if os.path.isfile(f):
                            os.remove(f)
                        elif os.path.isdir(f):
                            shutil.rmtree(f)
                except Exception as e:
                    print(f"清理 {f} 失败: {e}")
        else:
            print(f"⚠️ 找不到序列 '{jobname}' 的预测结果")

    print(f"\n[完成] 最终保留了 {output_count} 个标准 PDB 文档。")
    print("="*40)

# 执行整理与清理
finalize_and_cleanup(queries, result_dir)

2026-03-16 13:37:12,378 More than one sequence in /content/drive/MyDrive/input_fasta/output_part1.fasta, ignoring all but the first sequence
2026-03-16 13:37:21,065 Running on GPU
2026-03-16 13:37:21,583 Found 5 citations for tools or databases
2026-03-16 13:37:21,584 Query 1/1: output_part1 (length 40)


COMPLETE: 100%|██████████| 150/150 [elapsed: 00:01 remaining: 00:00]


2026-03-16 13:37:23,389 Setting max_seq=65, max_extra_seq=1
2026-03-16 13:37:56,302 alphafold2_ptm_model_1_seed_000 recycle=0 pLDDT=57.8 pTM=0.313
2026-03-16 13:38:13,877 alphafold2_ptm_model_1_seed_000 recycle=1 pLDDT=78.6 pTM=0.505 tol=1.76
2026-03-16 13:38:14,448 alphafold2_ptm_model_1_seed_000 recycle=2 pLDDT=82.1 pTM=0.538 tol=0.324
2026-03-16 13:38:15,016 alphafold2_ptm_model_1_seed_000 recycle=3 pLDDT=82.5 pTM=0.542 tol=0.161
2026-03-16 13:38:15,017 alphafold2_ptm_model_1_seed_000 took 37.6s (3 recycles)
2026-03-16 13:38:15,606 alphafold2_ptm_model_2_seed_000 recycle=0 pLDDT=52.2 pTM=0.237
2026-03-16 13:38:16,172 alphafold2_ptm_model_2_seed_000 recycle=1 pLDDT=52.1 pTM=0.247 tol=0.879
2026-03-16 13:38:16,737 alphafold2_ptm_model_2_seed_000 recycle=2 pLDDT=51.8 pTM=0.253 tol=2.97
2026-03-16 13:38:17,302 alphafold2_ptm_model_2_seed_000 recycle=3 pLDDT=52 pTM=0.241 tol=3.02
2026-03-16 13:38:17,303 alphafold2_ptm_model_2_seed_000 took 2.3s (3 recycles)
2026-03-16 13:38:17,885 alphaf

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import torch
import os

# 定义你的文件路径
file_path = "/content/drive/MyDrive/result/input_fasta_unrelaxed_rank_001_alphafold2_ptm_model_4_seed_000_map.pt"

def print_mapping_details(path):
    # 1. 检查挂载与文件是否存在
    if not os.path.exists(path):
        print(f"❌ 找不到文件！请检查：")
        print(f"   1. 是否运行了 drive.mount('/content/drive')")
        print(f"   2. 路径是否准确：{path}")
        return

    # 2. 加载张量
    try:
        mapping = torch.load(path)
    except Exception as e:
        print(f"读取文件时出错: {e}")
        return

    # 3. 解析信息
    num_atoms = mapping.shape[0]
    num_residues = mapping.max().item() + 1

    print("="*50)
    print(f"✅ 映射文件读取成功")
    print(f"📂 路径: .../{os.path.basename(path)}")
    print("-" * 50)
    print(f"🔸 总原子数 (Atoms): {num_atoms}")
    print(f"🔸 总残基数 (Residues): {num_residues}")
    print(f"🔸 平均每个残基包含原子数: {num_atoms/num_residues:.2f}")
    print("-" * 50)

    # 4. 展示详细映射关系
    print("📍 前 20 个原子的分配关系 [Atom Index -> Residue Index]:")
    # 格式化打印，每行显示 5 个以节省空间
    display_limit = min(20, num_atoms)
    for i in range(0, display_limit, 5):
        line = "  ".join([f"A{j:03d}→R{mapping[j].item():02d}" for j in range(i, min(i+5, display_limit))])
        print(f"   {line}")

    # 5. 验证第一个和最后一个原子
    print("\n🔍 边界检查:")
    print(f"   第一个原子 (Index 0)   属于残基: {mapping[0].item()}")
    print(f"   最后一个原子 (Index {num_atoms-1}) 属于残基: {mapping[-1].item()}")
    print("="*50)

# 执行打印
print_mapping_details(file_path)

✅ 映射文件读取成功
📂 路径: .../input_fasta_unrelaxed_rank_001_alphafold2_ptm_model_4_seed_000_map.pt
--------------------------------------------------
🔸 总原子数 (Atoms): 293
🔸 总残基数 (Residues): 40
🔸 平均每个残基包含原子数: 7.33
--------------------------------------------------
📍 前 20 个原子的分配关系 [Atom Index -> Residue Index]:
   A000→R00  A001→R00  A002→R00  A003→R00  A004→R01
   A005→R01  A006→R01  A007→R01  A008→R01  A009→R01
   A010→R02  A011→R02  A012→R02  A013→R02  A014→R03
   A015→R03  A016→R03  A017→R03  A018→R03  A019→R03

🔍 边界检查:
   第一个原子 (Index 0)   属于残基: 0
   最后一个原子 (Index 292) 属于残基: 39


# Instructions <a name="Instructions"></a>
**Quick start**
1. Upload your single fasta files to a folder in your Google Drive
2. Define path to the fold containing the fasta files (`input_dir`) define an outdir (`output_dir`)
3. Press "Runtime" -> "Run all".

**Result zip file contents**

At the end of the job a all results `jobname.result.zip` will be uploaded to your (`output_dir`) Google Drive. Each zip contains one protein.

1. PDB formatted structures sorted by avg. pIDDT. (unrelaxed and relaxed if `use_amber` is enabled).
2. Plots of the model quality.
3. Plots of the MSA coverage.
4. Parameter log file.
5. A3M formatted input MSA.
6. BibTeX file with citations for all used tools and databases.


**Troubleshooting**
* Check that the runtime type is set to GPU at "Runtime" -> "Change runtime type".
* Try to restart the session "Runtime" -> "Factory reset runtime".
* Check your input sequence.

**Known issues**
* Google Colab assigns different types of GPUs with varying amount of memory. Some might not have enough memory to predict the structure for a long sequence.
* Google Colab assigns different types of GPUs with varying amount of memory. Some might not have enough memory to predict the structure for a long sequence.
* Your browser can block the pop-up for downloading the result file. You can choose the `save_to_google_drive` option to upload to Google Drive instead or manually download the result file: Click on the little folder icon to the left, navigate to file: `jobname.result.zip`, right-click and select \"Download\" (see [screenshot](https://pbs.twimg.com/media/E6wRW2lWUAEOuoe?format=jpg&name=small)).

**Limitations**
* Computing resources: Our MMseqs2 API can handle ~20-50k requests per day.
* MSAs: MMseqs2 is very precise and sensitive but might find less hits compared to HHblits/HMMer searched against BFD or Mgnify.
* We recommend to additionally use the full [AlphaFold2 pipeline](https://github.com/deepmind/alphafold).

**Description of the plots**
*   **Number of sequences per position** - We want to see at least 30 sequences per position, for best performance, ideally 100 sequences.
*   **Predicted lDDT per position** - model confidence (out of 100) at each position. The higher the better.
*   **Predicted Alignment Error** - For homooligomers, this could be a useful metric to assess how confident the model is about the interface. The lower the better.

**Bugs**
- If you encounter any bugs, please report the issue to https://github.com/sokrypton/ColabFold/issues

**License**

The source code of ColabFold is licensed under [MIT](https://raw.githubusercontent.com/sokrypton/ColabFold/main/LICENSE). Additionally, this notebook uses AlphaFold2 source code and its parameters licensed under [Apache 2.0](https://raw.githubusercontent.com/deepmind/alphafold/main/LICENSE) and  [CC BY 4.0](https://creativecommons.org/licenses/by-sa/4.0/) respectively. Read more about the AlphaFold license [here](https://github.com/deepmind/alphafold).

**Acknowledgments**
- We thank the AlphaFold team for developing an excellent model and open sourcing the software.

- Do-Yoon Kim for creating the ColabFold logo.

- A colab by Sergey Ovchinnikov ([@sokrypton](https://twitter.com/sokrypton)), Milot Mirdita ([@milot_mirdita](https://twitter.com/milot_mirdita)) and Martin Steinegger ([@thesteinegger](https://twitter.com/thesteinegger)).
